In [2]:
import pandas as pd
import requests

In [6]:
# Pulling live S&P 500 stocks from the wikipedia
URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Wikipedia rejects request without no user- agent header so setting explicitly
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(URL, headers = headers)
response.raise_for_status() # fail loudly if page doesn't load

tables = pd.read_html(response.text)
sp500 = tables[0] # First table on the page

sp500 = sp500.rename(columns = {
    'Symbol' : 'symbol',
    'GICS Sector' : 'sector',
    'Date added' : 'date_added',
})

sp500['date_added'] = pd.to_datetime(sp500['date_added'], errors = 'coerce')
sp500 = sp500[['symbol', 'sector', 'date_added']]

print(f'Total constituents pulled from wikipedia : {len(sp500)}')


# filtering out names that have added too recently so that each stock has enogh trading history in the specified window

cutoff_date = pd.Timestamp('2020-01-01')
eligible = sp500[sp500['date_added'] <= cutoff_date].copy()
print(f'Total eligible constituents : {len(eligible)}')

# Stratified Random Sampling across GICS Sector. Used largest - Remainder (Hamilton) Method.
target_n = 50
seed = 42

sector_counts = eligible['sector'].value_counts()
sector_weight = sector_counts / sector_counts.sum()
raw_alloc = sector_weight * target_n

alloc = raw_alloc.apply(int) # Gurrantees whole share
remaining = target_n - alloc.sum() # unassigned slot
remainders = raw_alloc - alloc
largest = remainders.sort_values( ascending = False)

for sector in largest.index[ : remaining]:
    alloc[sector] +=1 

print("\nSector allocation (largest-remainder method):")
print(alloc.sort_values(ascending=False))
print(f"Sum check: {alloc.sum()} (target was {target_n})")

# Random draw within each sector, using the fixed seed
picked = []
for sector, n in alloc.items():
    pool = eligible[eligible['sector'] == sector]
    n = min(n, len(pool))
    picked.append(pool.sample(n=n, random_state = seed))

sample = pd.concat(picked).sort_values(['sector', 'symbol']).reset_index(drop=True)
 
print(f"\nFinal drawn universe: {len(sample)} tickers\n")
print(sample.to_string(index=False))
 
sample['yf_symbol'] = sample['symbol'].str.replace('.', '-', regex=False)
sample.to_csv('stratified_50_universe.csv', index=False)
print("\nSaved to stratified_50_universe.csv")





C:\Users\sentr\AppData\Local\Temp\ipykernel_21516\973339296.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Total constituents pulled from wikipedia : 503
Total eligible constituents : 408

Sector allocation (largest-remainder method):
sector
Industrials               8
Financials                8
Health Care               6
Information Technology    6
Consumer Discretionary    5
Consumer Staples          4
Utilities                 3
Real Estate               3
Materials                 3
Communication Services    2
Energy                    2
Name: count, dtype: int64
Sum check: 50 (target was 50)

Final drawn universe: 50 tickers

symbol                 sector date_added
    EA Communication Services 2002-07-22
 GOOGL Communication Services 2006-04-03
  BKNG Consumer Discretionary 2009-11-06
    GM Consumer Discretionary 2013-06-06
  ROST Consumer Discretionary 2009-12-21
   TPR Consumer Discretionary 2004-09-01
  ULTA Consumer Discretionary 2016-04-18
   KMB       Consumer Staples 1957-03-04
    KR       Consumer Staples 1957-03-04
    PM       Consumer Staples 2008-03-31
   TGT       Co